# AuralGuard — Kaggle Training Pipeline (Full)
Trains B1 → B2 → B3 → B5 → AuralGuard with:
- **Checkpoint recovery** — resume from last epoch if session crashes
- **HuggingFace sync** — pull/push checkpoints across accounts & sessions
- **All datasets** — zero-shot eval + augmentation corpora
- **Auto-upload** — figures, metrics, checkpoints pushed to HF Hub

**Setup:**
1. Add datasets via **+Data** (right sidebar):
   - `awsaf49/asvpoof-2019-dataset` (training, required)
   - `abdallamohamed312/in-the-wild-audio-deepfake`
   - `walimuhammadahmad/fakeaudio` (WaveFake)
   - `kkijjaa/asvspoof2021-la`
   - `serjkalinovskiy/asvspoof2021-df`
   - `riosgonzalo/musan-rirs`
   - `nhattruongdev/rirs-noises`
2. Enable **Internet** + **GPU** in notebook Settings
3. Run cells sequentially

In [ ]:
# ── 1. Progress & logging helpers ──
import time, subprocess, shlex, json, os
from datetime import datetime
from pathlib import Path

# ── Progress tracking ──
PROGRESS_FILE = Path("/kaggle/working/auralguard/progress.json")
PROGRESS_FILE.parent.mkdir(parents=True, exist_ok=True)

def mark_done(cell_id):
    """Mark a cell as completed. Saves to progress file."""
    progress = {}
    if PROGRESS_FILE.exists():
        try:
            progress = json.loads(PROGRESS_FILE.read_text())
        except Exception:
            progress = {}
    progress[cell_id] = {
        "done": True,
        "time": datetime.now().isoformat(),
    }
    PROGRESS_FILE.write_text(json.dumps(progress, indent=2))
    log(f"[progress] {cell_id} marked DONE")

def reset_training_progress():
    """Clear training entries from progress if no checkpoint exists."""
    if not PROGRESS_FILE.exists():
        return
    progress = json.loads(PROGRESS_FILE.read_text())
    model_map = {
        "train_b1": "b1_lcnn",
        "train_b2": "b2_rawnet2",
        "train_b3": "b3_aasist",
        "train_b5": "b5_wavlm_ocs",
        "train_auralguard": "auralguard",
    }
    always_reset = ["aug_manifests"]  # always rebuild these
    removed = []
    for key in always_reset:
        if key in progress:
            del progress[key]
            removed.append(key)
    for key, exp_name in model_map.items():
        if key in progress:
            ckpt = Path(f"experiments/{exp_name}/checkpoints/best.ckpt")
            if not ckpt.exists():
                del progress[key]
                removed.append(key)
    if removed:
        PROGRESS_FILE.write_text(json.dumps(progress, indent=2))
        log(f"[progress] Reset stale entries: {removed}")

def get_progress():
    """Load progress from file."""
    if PROGRESS_FILE.exists():
        try:
            return json.loads(PROGRESS_FILE.read_text())
        except Exception:
            return {}
    return {}

# ── Logging helpers ──
def ts(): return datetime.now().strftime("[%H:%M:%S]")
def log(msg): print(f"{ts()} {msg}", flush=True)

def run_cmd(cmd, timeout_min=None, cwd=None, label=""):
    log(f"$ {cmd}")
    start = time.time()
    last_heartbeat = time.time()
    proc = subprocess.Popen(
        cmd if isinstance(cmd, list) else shlex.split(cmd),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        cwd=cwd, text=True, bufsize=1
    )
    last_output = time.time()
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
        if line.strip(): last_output = time.time()
        now = time.time()
        if now - last_heartbeat > 120 and now - last_output > 60:
            elapsed_min = (now - start) / 60
            tag = f" [{label}]" if label else ""
            log(f"[heartbeat{tag}] Still running... {elapsed_min:.0f} min elapsed")
            last_heartbeat = now
        if timeout_min and (now - start) > timeout_min * 60:
            proc.kill()
            log(f"TIMEOUT after {timeout_min} min -- aborting")
            return False
    proc.wait()
    elapsed = time.time() - start
    ok = proc.returncode == 0
    status = "OK" if ok else f"FAILED (code {proc.returncode})"
    log(f"Done in {elapsed:.0f}s -- {status}")
    return ok

try:
    from tqdm.auto import tqdm
except ImportError:
    !pip install tqdm -q
    from tqdm.auto import tqdm

log("Helpers loaded")

In [ ]:
# ── 1b. Progress status (auto-run on every session start) ──
# Shows exactly which cells are done and which to run next.

CELL_REGISTRY = {
    "helpers":              "1.  Helpers & logging",
    "hf_setup":             "2.  HuggingFace Hub setup",
    "checkpoint_aids":      "3.  Checkpoint recovery helpers",
    "find_dataset":         "4a. Find ASVspoof dataset",
    "zero_shot_data":       "4b. Download zero-shot datasets",
    "augmentation_data":    "4c. Download augmentation corpora",
    "install":              "4d. Install AuralGuard",
    "hf_login":             "5.  HF Login + create repo",
    "build_manifests":      "6.  Build manifests",
    "aug_manifests":        "7.  Build augmentation manifests",
    "train_b1":             "8a. Train B1 (LFCC-LCNN)",
    "train_b2":             "8b. Train B2 (RawNet2)",
    "train_b3":             "8c. Train B3 (AASIST)",
    "train_b5":             "8d. Train B5 (WavLM+AASIST)",
    "train_auralguard":     "8e. Train AuralGuard",
    "eval_in_domain":       "9.  In-domain evaluation",
    "eval_zero_shot":       "10. Zero-shot evaluation",
    "figures":              "11. Generate paper figures",
    "results_table":        "12. Export results table",
    "hf_upload_all":        "13. Upload results to HF Hub",
    "model_deploy":         "14. Upload model for deployment",
    "package":              "15. Package for local download",
}

progress = get_progress()

done_ids = {k for k, v in progress.items() if v.get("done")}
pending_ids = [k for k in CELL_REGISTRY if k not in done_ids]

print("\n" + "=" * 60)
print("  AURALGUARD PROGRESS STATUS")
print("=" * 60)
print(f"  {"ID":<22} │ Status")
print("─" * 60)
for cid, label in CELL_REGISTRY.items():
    if cid in done_ids:
        print(f"  {label:<22} │ ✅ DONE")
    else:
        print(f"  {label:<22} │ ⏳ pending")
print("─" * 60)
if pending_ids:
    next_id = pending_ids[0]
    print(f"  ➜  NEXT: Run cell for \"{CELL_REGISTRY[next_id]}\"")
else:
    print("  🎉 ALL CELLS COMPLETE")
print("=" * 60)

mark_done("helpers")

In [ ]:
# ── 2. HuggingFace Hub setup (token + repo helpers) ──
# This cell provides hf_login(), hf_upload(), hf_download(), hf_repo_exists()
# All other cells use these helpers for cross-account checkpoint sync.

import os, json
from pathlib import Path

# HF repo where checkpoints + results + figures are stored
HF_REPO = "MoshinAli/auralguard-checkpoints"

def hf_login():
    """Login to HuggingFace Hub. Tries: Kaggle Secrets > env var > .env > prompt."""
    token = None

    # 1. Try Kaggle Secrets (user added HF_TOKEN there)
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        token = secrets.get_secret("HF_TOKEN")
        log("Loaded HF token from Kaggle Secrets")
    except Exception:
        pass

    # 2. Try environment variables
    if not token:
        token = os.environ.get("HF_TOKEN") or os.environ.get("HF")
        if token:
            log("Loaded HF token from environment variable")

    # 3. Try .env file in repo root
    if not token:
        env_file = Path("/kaggle/working/auralguard/.env")
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                if line.startswith("HF="):
                    token = line.split("=", 1)[1].strip()
                    log("Loaded HF token from .env file")
                    break

    # 4. Prompt as last resort
    if not token:
        token = input("Enter HuggingFace token (hf_xxx): ").strip()

    assert token, "No HuggingFace token provided"
    from huggingface_hub import login
    login(token=token)
    log(f"HuggingFace logged in")
    return token

def hf_repo_exists(repo_id=None):
    """Check if the HF repo exists."""
    from huggingface_hub import HfApi
    repo_id = repo_id or HF_REPO
    try:
        HfApi().repo_info(repo_id)
        return True
    except Exception:
        return False

def hf_create_repo(repo_id=None):
    """Create the HF repo if it doesn't exist."""
    from huggingface_hub import HfApi
    repo_id = repo_id or HF_REPO
    if not hf_repo_exists(repo_id):
        HfApi().create_repo(repo_id, repo_type="model", exist_ok=True)
        log(f"Created HF repo: {repo_id}")
    else:
        log(f"HF repo exists: {repo_id}")

def hf_upload(local_path, repo_path, repo_id=None):
    """Upload a file to HF Hub. Skips if already exists with same size."""
    from huggingface_hub import HfApi
    repo_id = repo_id or HF_REPO
    local = Path(local_path)
    if not local.exists():
        log(f"  [skip] {local} not found")
        return False
    api = HfApi()
    # Check if file already exists
    try:
        info = api.repo_info(repo_id)
        for s in (info.siblings or []):
            if s.rfilename == repo_path and s.size == local.stat().st_size:
                log(f"  [skip] {repo_path} (same size)")
                return True
    except Exception:
        pass
    api.upload_file(path_or_fileobj=str(local), path_in_repo=repo_path, repo_id=repo_id)
    log(f"  [uploaded] {repo_path}")
    return True

def hf_download(repo_path, local_path, repo_id=None):
    """Download a file from HF Hub. Returns True if downloaded."""
    from huggingface_hub import hf_hub_download
    repo_id = repo_id or HF_REPO
    local = Path(local_path)
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        hf_hub_download(repo_id=repo_id, filename=repo_path, local_dir=str(local.parent))
        log(f"  [downloaded] {repo_path}")
        return True
    except Exception as e:
        log(f"  [skip] {repo_path}: {e}")
        return False

def hf_list_files(repo_id=None):
    """List all files in the HF repo."""
    from huggingface_hub import HfApi
    repo_id = repo_id or HF_REPO
    try:
        files = HfApi().list_repo_files(repo_id)
        log(f"HF repo has {len(files)} files")
        return files
    except Exception as e:
        log(f"Could not list HF repo: {e}")
        return []

log("HF helpers loaded")
mark_done("hf_setup")

In [ ]:
# ── 3. Checkpoint recovery helpers ──
# Provides: find_checkpoint(), save_checkpoint(), sync_checkpoint_from_hf()

from pathlib import Path

def find_checkpoint(exp_name, experiments_dir="experiments"):
    """Find existing checkpoint for an experiment.
    Priority: best.ckpt (local) > last.ckpt (local) > best.ckpt (HF) > None
    """
    exp_dir = Path(experiments_dir) / exp_name
    ckpt_dir = exp_dir / "checkpoints"
    
    # Check local first
    best = ckpt_dir / "best.ckpt"
    last = ckpt_dir / "last.ckpt"
    
    if best.exists():
        ckpt = torch.load(str(best), map_location="cpu", weights_only=False)
        epoch = ckpt.get("epoch", -1)
        eer = ckpt.get("dev_eer", float("inf"))
        log(f"  [local] {exp_name}: best.ckpt (epoch={epoch}, EER={eer:.4f})")
        return str(best), ckpt
    
    if last.exists():
        ckpt = torch.load(str(last), map_location="cpu", weights_only=False)
        epoch = ckpt.get("epoch", -1)
        log(f"  [local] {exp_name}: last.ckpt (epoch={epoch})")
        return str(last), ckpt
    
    # Try pulling from HF
    log(f"  [hf] Trying to pull {exp_name} from HuggingFace...")
    for fname in ["best.ckpt", "last.ckpt"]:
        repo_path = f"checkpoints/{exp_name}/{fname}"
        if hf_download(repo_path, str(ckpt_dir / fname)):
            ckpt = torch.load(str(ckpt_dir / fname), map_location="cpu", weights_only=False)
            epoch = ckpt.get("epoch", -1)
            eer = ckpt.get("dev_eer", float("inf"))
            log(f"  [hf] {exp_name}: {fname} (epoch={epoch}, EER={eer:.4f})")
            return str(ckpt_dir / fname), ckpt
    
    log(f"  [none] {exp_name}: no checkpoint found")
    return None, None

def save_checkpoint(model, cfg, epoch, eer, exp_name, experiments_dir="experiments"):
    """Save checkpoint locally and upload to HF Hub."""
    import torch
    from omegaconf import OmegaConf
    exp_dir = Path(experiments_dir) / exp_name
    ckpt_dir = exp_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        cfg_container = OmegaConf.to_container(cfg, resolve=True)
    except Exception:
        cfg_container = dict(cfg)
    
    state = {
        "model": model.state_dict(),
        "cfg": cfg_container,
        "epoch": epoch,
        "dev_eer": eer,
    }
    
    # Save locally
    best_path = ckpt_dir / "best.ckpt"
    last_path = ckpt_dir / "last.ckpt"
    torch.save(state, str(best_path))
    torch.save(state, str(last_path))
    log(f"  Saved {exp_name} best.ckpt (epoch={epoch}, EER={eer:.4f})")
    
    # Upload to HF (non-blocking)
    try:
        hf_upload(best_path, f"checkpoints/{exp_name}/best.ckpt")
        hf_upload(last_path, f"checkpoints/{exp_name}/last.ckpt")
    except Exception as e:
        log(f"  [warn] HF upload failed: {e}")
    
    return str(best_path)

def get_resume_epoch(exp_name, experiments_dir="experiments"):
    """Get the epoch to resume from for an experiment."""
    ckpt_path, ckpt = find_checkpoint(exp_name, experiments_dir)
    if ckpt is not None:
        return ckpt.get("epoch", -1) + 1
    return 0

log("Checkpoint recovery helpers loaded")
mark_done("checkpoint_aids")

## 4. Dataset Setup
Finds ASVspoof 2019 LA + downloads zero-shot eval datasets + augmentation corpora.

In [ ]:
# ── 4a. Find ASVspoof 2019 LA dataset ──
import os

log("Searching for ASVspoof 2019 dataset...")
INPUT_DIR = None

EXACT_PATHS = [
    # Modern Kaggle: /kaggle/input/{slug}/  (double LA/LA nesting)
    Path("/kaggle/input/asvpoof-2019-dataset/LA/LA"),
    Path("/kaggle/input/asvpoof-2019-dataset/LA"),
    Path("/kaggle/input/asvpoof-2019-dataset"),
    # Old Kaggle: /kaggle/input/datasets/{owner}/{slug}/
    Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA"),
    Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA"),
    Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset"),
]
for p in EXACT_PATHS:
    if (p / "ASVspoof2019_LA_cm_protocols").exists():
        INPUT_DIR = p
        log(f"  Found at {p}")
        break

if INPUT_DIR is None:
    for root in [Path("/kaggle/input/asvpoof-2019-dataset"),
                 Path("/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset"),
                 Path("/kaggle/input/asvpoof2019dataset"),
                 Path("/kaggle/input/datasets/awsaf49/asvpoof2019dataset")]:
        if not root.exists():
            continue
        for sub in ["", "LA", "LA/LA", "ASVspoof 2019 Dataset", "ASVspoof 2019 Dataset/LA", "ASVspoof 2019 Dataset/LA/LA"]:
            candidate = root / sub if sub else root
            proto = candidate / "ASVspoof2019_LA_cm_protocols"
            if candidate.exists() and proto.exists():
                INPUT_DIR = candidate
                log(f"  Matched at {candidate}")
                break
        if INPUT_DIR:
            break

# Glob search across all of /kaggle/input/ as last resort
if INPUT_DIR is None:
    log("Trying glob search across /kaggle/input/...")
    for hit in Path("/kaggle/input").rglob("ASVspoof2019_LA_cm_protocols"):
        candidate = hit.parent
        log(f"  Found via glob: {candidate}")
        INPUT_DIR = candidate
        break

if INPUT_DIR is None:
    log("ERROR: Dataset not found!")
    log("  Listing /kaggle/input contents:")
    for p in Path("/kaggle/input").iterdir():
        log(f"    {p.name}  dir={p.is_dir()}")
    for hit in Path("/kaggle/input").rglob("*cm_protocols*"):
        log(f"    FOUND: {hit}")
    log("  -> Click +Data in right sidebar, search 'awsaf49/asvpoof-2019-dataset', click Add")
    raise FileNotFoundError("Dataset not found")

expected = ["ASVspoof2019_LA_train", "ASVspoof2019_LA_dev",
            "ASVspoof2019_LA_eval", "ASVspoof2019_LA_cm_protocols"]
for d in expected:
    found = (INPUT_DIR / d).exists()
    log(f"  {d}: {'OK' if found else 'MISSING!'}")

RAW = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
if not RAW.exists():
    RAW.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(INPUT_DIR), str(RAW), target_is_directory=True)
    log(f"Symlink: {RAW} -> {INPUT_DIR}")
else:
    log("Symlink exists")

flac_count = len(list(RAW.glob("ASVspoof2019_LA_train/flac/*.flac")))
log(f"Train FLAC files: {flac_count}")
assert flac_count > 1000, f"Too few files ({flac_count})"
log("ASVspoof 2019 LA ready")
mark_done("find_dataset")

In [ ]:
# ── 4b. Download zero-shot evaluation datasets ──
# Checks Kaggle inputs first (instant mount), falls back to download.
#
# Kaggle input datasets (added via +Data):
#   1. awsaf49/asvpoof-2019-dataset
#   2. abdallamohamed312/in-the-wild-audio-deepfake
#   3. walimuhammadahmad/fakeaudio (WaveFake)
#   4. kkijjaa/asvspoof2021-la
#   5. serjkalinovskiy/asvspoof2021-df
#   6. riosgonzalo/musan-rirs
#   7. nhattruongdev/rirs-noises
#   If not added, the notebook downloads them automatically (slower).

DATA_RAW = Path("/kaggle/working/data/raw")

# Exact Kaggle input paths (verified)
ZERO_SHOT_DATASETS = {
    "in_the_wild": {
        "name": "In-the-Wild",
        "url": "https://huggingface.co/datasets/mueller91/In-The-Wild/resolve/main/release_in_the_wild.zip",
        "dir": DATA_RAW / "in_the_wild",
        "extract": "zip",
        "subdir": "release_in_the_wild",
        "kaggle_inputs": [
            Path("/kaggle/input/in-the-wild-audio-deepfake"),
            Path("/kaggle/input/datasets/abdallamohamed312/in-the-wild-audio-deepfake"),
        ],
    },
    "wavefake": {
        "name": "WaveFake",
        "url": "https://zenodo.org/records/5642694/files/generated_audio.zip?download=1",
        "dir": DATA_RAW / "wavefake",
        "extract": "zip",
        "subdir": "generated_audio",
        "kaggle_inputs": [
            Path("/kaggle/input/fakeaudio"),
            Path("/kaggle/input/datasets/walimuhammadahmad/fakeaudio"),
        ],
    },
    "asvspoof2021_la": {
        "name": "ASVspoof2021 LA",
        "url": "https://zenodo.org/records/4837263/files/ASVspoof2021_LA_eval.tar.gz?download=1",
        "dir": DATA_RAW / "ASVspoof2021_LA",
        "extract": "tar",
        "subdir": None,
        "kaggle_inputs": [
            Path("/kaggle/input/asvspoof2021-la"),
            Path("/kaggle/input/datasets/kkijjaa/asvspoof2021-la"),
        ],
    },
    "asvspoof2021_df": {
        "name": "ASVspoof2021 DF",
        "urls": [
            "https://zenodo.org/records/4835108/files/ASVspoof2021_DF_eval_part00.tar.gz?download=1",
            "https://zenodo.org/records/4835108/files/ASVspoof2021_DF_eval_part01.tar.gz?download=1",
            "https://zenodo.org/records/4835108/files/ASVspoof2021_DF_eval_part02.tar.gz?download=1",
            "https://zenodo.org/records/4835108/files/ASVspoof2021_DF_eval_part03.tar.gz?download=1",
        ],
        "dir": DATA_RAW / "ASVspoof2021_DF",
        "extract": "tar",
        "subdir": None,
        "kaggle_inputs": [
            Path("/kaggle/input/asvspoof2021-df"),
            Path("/kaggle/input/datasets/serjkalinovskiy/asvspoof2021-df"),
        ],
    },
}

import zipfile, tarfile, shutil
from urllib.request import urlopen, Request

def find_kaggle_input(candidates):
    """Check exact Kaggle input paths for a matching dataset directory.
    Also checks for common nested subdirectories for ASVspoof 2021 datasets.
    NOTE: In-the-Wild is NOT checked here because meta.csv lives at the root.
    """
    for path in candidates:
        if isinstance(path, str):
            path = Path(path)
        if path.exists() and any(path.iterdir()):
            log(f"  [kaggle] Found at {path}")
            return path
        # Check nested subdirs for ASVspoof 2021 (double eval nesting)
        # Do NOT add release_in_the_wild here — meta.csv is at the root!
        for sub in ["ASVspoof2021_LA_eval", "ASVspoof2021_DF_eval",
                     "ASVspoof2021-DF", "generated_audio", "RIRS_NOISES"]:
            candidate = path / sub
            if candidate.exists() and any(candidate.iterdir()):
                log(f"  [kaggle] Found at {candidate} (nested in {path.name})")
                return candidate
    return None

def download_file(url, dest, label):
    if dest.exists():
        if dest.suffix == ".zip":
            try:
                with zipfile.ZipFile(str(dest)) as zf:
                    zf.testzip()
                log(f"  [skip] {label} already at {dest} (valid zip)")
                return
            except (zipfile.BadZipFile, Exception) as e:
                log(f"  [warn] {label} archive corrupted ({e}), re-downloading...")
                dest.unlink(missing_ok=True)
        else:
            log(f"  [skip] {label} already at {dest}")
            return
    log(f"  [dl] {label} ...")
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = Request(url, headers={"User-Agent": "Mozilla/5.0 (Kaggle Notebook)"})
    with urlopen(req) as resp:
        total = int(resp.headers.get("Content-Length", 0))
        dl = 0
        with open(str(dest), "wb") as f:
            while chunk := resp.read(1 << 20):
                f.write(chunk)
                dl += len(chunk)
                if total:
                    pct = dl / total * 100
                    if int(pct) % 10 == 0:
                        print(f"\r  [dl] {label} ... {pct:.0f}%", end="", flush=True)
    print()
    log(f"  [ok] {label} downloaded")

for key, info in ZERO_SHOT_DATASETS.items():
    log(f"--- {info['name']} ---")
    target = info["dir"]
    # Skip if already exists and has content
    if target.exists() and any(target.iterdir()):
        log(f"  [skip] already exists at {target}")
        continue
    
    # 1) Try Kaggle input (instant)
    kaggle_src = find_kaggle_input(info.get("kaggle_inputs", []))
    if kaggle_src:
        target.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(str(kaggle_src), str(target), target_is_directory=True)
        log(f"  [kaggle] Symlinked {target} -> {kaggle_src}")
        continue
    
    # 2) Fall back to download
    log(f"  [dl] Not found in Kaggle inputs, downloading...")
    urls = info.get("urls", [info["url"]])
    target.mkdir(parents=True, exist_ok=True)
    for i, url in enumerate(urls):
        ext = "zip" if info["extract"] == "zip" else "tar.gz"
        archive = DATA_RAW / f"{key}_part{i}.{ext}" if len(urls) > 1 else DATA_RAW / f"{key}.{ext}"
        download_file(url, archive, f"{info['name']} part {i+1}/{len(urls)}" if len(urls) > 1 else info["name"])
        log(f"  Extracting {archive.name}...")
        if info["extract"] == "zip":
            with zipfile.ZipFile(str(archive)) as zf:
                zf.extractall(str(DATA_RAW))
        else:
            with tarfile.open(str(archive)) as tf:
                tf.extractall(str(DATA_RAW))
        archive.unlink(missing_ok=True)
    
    # Move from subdir if needed
    if info["subdir"]:
        subdir = DATA_RAW / info["subdir"]
        if subdir.exists():
            for item in subdir.iterdir():
                dest_item = target / item.name
                if not dest_item.exists():
                    shutil.move(str(item), str(dest_item))
            shutil.rmtree(str(subdir))
    
    log(f"  [ok] {info['name']} ready at {target}")

log("Zero-shot datasets done")
mark_done("zero_shot_data")

In [ ]:
# ── 4c. Download augmentation corpora (MUSAN + RIRs) ──
# Same as above: checks Kaggle inputs first, falls back to download.

AUG_DATASETS = {
    "musan": {
        "name": "MUSAN (noise)",
        "url": "https://www.openslr.org/resources/17/musan.tar.gz",
        "dir": DATA_RAW / "musan",
        "extract": "tar",
        "kaggle_inputs": [
            Path("/kaggle/input/musan-rirs"),
            Path("/kaggle/input/datasets/riosgonzalo/musan-rirs"),
        ],
        "kaggle_subdir": "musan",  # musan/ lives inside the kaggle input
    },
    "rirs": {
        "name": "RIRs (SLR28)",
        "url": "https://www.openslr.org/resources/28/rirs_noises.zip",
        "dir": DATA_RAW / "RIRS_NOISES",
        "extract": "zip",
        "kaggle_inputs": [
            Path("/kaggle/input/rirs-noises"),
            Path("/kaggle/input/datasets/nhattruongdev/rirs-noises"),
            Path("/kaggle/input/musan-rirs"),
            Path("/kaggle/input/datasets/riosgonzalo/musan-rirs"),
        ],
        "kaggle_subdir": "RIRS_NOISES",  # RIRS_NOISES/ may be nested inside
    },
}

for key, info in AUG_DATASETS.items():
    log(f"--- {info['name']} ---")
    target = info["dir"]
    if target.exists() and any(target.iterdir()):
        log(f"  [skip] already exists at {target}")
        continue
    
    # 1) Try Kaggle input (instant)
    kaggle_src = find_kaggle_input(info.get("kaggle_inputs", []))
    if kaggle_src:
        # Check if there's a nested subdir we need to point to
        kaggle_subdir = info.get("kaggle_subdir")
        if kaggle_subdir:
            nested = kaggle_src / kaggle_subdir
            if nested.exists() and any(nested.iterdir()):
                kaggle_src = nested
                log(f"  [kaggle] Using nested subdir: {kaggle_src}")
        os.symlink(str(kaggle_src), str(target), target_is_directory=True)
        log(f"  [kaggle] Symlinked {target} -> {kaggle_src}")
        continue
    
    # 2) Fall back to download
    log(f"  [dl] Not found in Kaggle inputs, downloading...")
    ext = "zip" if info["extract"] == "zip" else "tar.gz"
    archive = DATA_RAW / f"{key}.{ext}"
    download_file(info["url"], archive, info["name"])
    
    log(f"  Extracting {info['name']}...")
    if info["extract"] == "zip":
        with zipfile.ZipFile(str(archive)) as zf:
            zf.extractall(str(DATA_RAW))
    else:
        with tarfile.open(str(archive)) as tf:
            tf.extractall(str(DATA_RAW))
    
    archive.unlink(missing_ok=True)
    log(f"  [ok] {info['name']} ready at {target}")

log("Augmentation datasets done")
mark_done("augmentation_data")

In [ ]:
# ── 4d. Install AuralGuard ──
log("Starting installation...")

REPO_DIR = Path("/kaggle/working/auralguard")
import shutil

# Always go to safe CWD first (in case REPO_DIR was deleted previously)
os.chdir("/kaggle/working")

# Check if repo already has the right structure
has_repo = (REPO_DIR / "scripts" / "train.py").exists()
if has_repo:
    log("Repo already present -- updating via git pull...")
    os.chdir(str(REPO_DIR))
    run_cmd("git pull", timeout_min=5)
else:
    if REPO_DIR.exists():
        log("Removing incomplete directory...")
        shutil.rmtree(str(REPO_DIR), ignore_errors=True)
    log("Cloning repo...")
    ok = run_cmd(
        "git clone https://github.com/MIHMahmudEli/auralguard.git /kaggle/working/auralguard",
        timeout_min=30,
        cwd="/kaggle/working"
    )
    if not ok:
        log("ERROR: Git clone failed. Possible causes:")
        log("  1. Internet is OFF in Kaggle settings (Settings -> Internet -> ON)")
        log("  2. GitHub is blocked / rate limited")
        log("  3. Repo URL is wrong (should be MIHMahmudEli/auralguard)")
        raise RuntimeError("Clone failed - see logs above")
    os.chdir(str(REPO_DIR))
log(f"Working dir: {os.getcwd()}")

# Verify scripts exist
scripts_dir = REPO_DIR / "scripts"
if not scripts_dir.exists():
    raise RuntimeError(f"Scripts dir not found at {scripts_dir}")
log(f"Scripts found: {[f.name for f in scripts_dir.iterdir()]}")

# Downgrade PyTorch for P100 (sm_60) support
run_cmd("pip install 'torch>=2.3,<2.4' 'torchaudio>=2.3,<2.4' --force-reinstall --index-url https://download.pytorch.org/whl/cu121", timeout_min=15)
run_cmd("pip install -e .[train,dev]", timeout_min=15)
run_cmd("pip install 'transformers>=4.41,<4.45' 'torchvision>=0.18,<0.19' --force-reinstall", timeout_min=15)
run_cmd("pip install datasets huggingface_hub tensorboard", timeout_min=10)

import torch
log(f"PyTorch {torch.__version__} -- CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    log(f"  GPU: {torch.cuda.get_device_name(0)}  Memory: {props.total_memory / 1e9:.1f} GB")
log("Installation complete")
mark_done("install")

In [ ]:
# ── 5. Login to HuggingFace + create repo ──
hf_login()
hf_create_repo()
mark_done("hf_login")

In [ ]:
# ── 6. Build manifests for ALL datasets ──
import pandas as pd

MANIFEST_DIR = Path("/kaggle/working/data/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = ["utt_id", "path", "label", "attack", "dataset", "lang", "split", "codec"]

# -- ASVspoof 2019 LA (train/dev/eval) --
RAW_2019 = Path("/kaggle/working/data/raw/ASVspoof2019_LA")
for split, suffix in [("train", "trn"), ("dev", "trl"), ("eval", "trl")]:
    proto = RAW_2019 / "ASVspoof2019_LA_cm_protocols" / f"ASVspoof2019.LA.cm.{split}.{suffix}.txt"
    audio_dir = RAW_2019 / f"ASVspoof2019_LA_{split}" / "flac"
    log(f"Building {split} manifest...")
    assert proto.exists(), f"Protocol not found: {proto}"
    assert audio_dir.exists(), f"Audio dir not found: {audio_dir}"
    lines = proto.read_text().splitlines()
    rows = []
    for line in tqdm(lines, desc=f"{split}"):
        parts = line.split()
        utt, attack, key = parts[1], parts[3], parts[4]
        rows.append({
            "utt_id": utt, "path": str(audio_dir / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2019_la", "lang": "en", "split": split, "codec": "none",
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    out = MANIFEST_DIR / f"asvspoof2019_la_{split}.csv"
    df.to_csv(out, index=False)
    log(f"  {out.name}: {len(df)} rows ({df.label.sum()} spoof)")

# -- ASVspoof 2021 LA --
ROOT_2021LA = DATA_RAW / "ASVspoof2021_LA"
# Handle nested ASVspoof2021_LA_eval/ directory from Kaggle
if ROOT_2021LA.exists():
    for sub in ["ASVspoof2021_LA_eval", "ASVspoof2021_LA"]:
        candidate = ROOT_2021LA / sub
        if candidate.exists() and (candidate / "flac").exists():
            ROOT_2021LA = candidate
            log(f"  ASVspoof 2021 LA nested at: {ROOT_2021LA}")
            break

# Try trial_metadata.txt first, then fallback to CM protocol file
trial_meta_2021la = ROOT_2021LA / "trial_metadata.txt"
cm_proto_2021la = ROOT_2021LA / "ASVspoof2021.LA.cm.eval.trl.txt"
if ROOT_2021LA.exists() and trial_meta_2021la.exists():
    log("Building ASVspoof 2021 LA manifest (from trial_metadata.txt)...")
    rows = []
    for line in trial_meta_2021la.read_text().splitlines():
        parts = line.split()
        if len(parts) < 8: continue
        spk, utt, codec, _ch, attack, key, _trim, _sub = parts[:8]
        rows.append({
            "utt_id": utt, "path": str(ROOT_2021LA / "flac" / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2021_la", "lang": "en", "split": "eval", "codec": codec,
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(MANIFEST_DIR / "asvspoof2021_la_eval.csv", index=False)
    log(f"  asvspoof2021_la_eval.csv: {len(df)} rows")
elif ROOT_2021LA.exists() and cm_proto_2021la.exists():
    log("Building ASVspoof 2021 LA manifest (from CM protocol file)...")
    rows = []
    for line in cm_proto_2021la.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        spk, utt, attack, _dash, key = parts[:5]
        rows.append({
            "utt_id": utt, "path": str(ROOT_2021LA / "flac" / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2021_la", "lang": "en", "split": "eval", "codec": "unknown",
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(MANIFEST_DIR / "asvspoof2021_la_eval.csv", index=False)
    log(f"  asvspoof2021_la_eval.csv: {len(df)} rows")

# -- ASVspoof 2021 DF --
ROOT_2021DF = DATA_RAW / "ASVspoof2021_DF"
# Handle nested directories from Kaggle
if ROOT_2021DF.exists():
    for sub in ["ASVspoof2021_DF_eval", "ASVspoof2021-DF", "ASVspoof2021_DF"]:
        candidate = ROOT_2021DF / sub
        if candidate.exists() and (candidate / "flac").exists():
            ROOT_2021DF = candidate
            log(f"  ASVspoof 2021 DF nested at: {ROOT_2021DF}")
            break

# Try trial_metadata.txt first, then fallback to CM protocol file
trial_meta_2021df = ROOT_2021DF / "trial_metadata.txt"
cm_proto_2021df = ROOT_2021DF / "ASVspoof2021.DF.cm.eval.trl.txt"
if ROOT_2021DF.exists() and trial_meta_2021df.exists():
    log("Building ASVspoof 2021 DF manifest (from trial_metadata.txt)...")
    rows = []
    for line in trial_meta_2021df.read_text().splitlines():
        parts = line.split()
        if len(parts) < 7: continue
        spk, utt, codec, _src, attack, key = parts[:6]
        rows.append({
            "utt_id": utt, "path": str(ROOT_2021DF / "flac" / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2021_df", "lang": "en", "split": "eval", "codec": codec,
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(MANIFEST_DIR / "asvspoof2021_df_eval.csv", index=False)
    log(f"  asvspoof2021_df_eval.csv: {len(df)} rows")
elif ROOT_2021DF.exists() and cm_proto_2021df.exists():
    log("Building ASVspoof 2021 DF manifest (from CM protocol file)...")
    rows = []
    for line in cm_proto_2021df.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5: continue
        spk, utt, attack, _dash, key = parts[:5]
        rows.append({
            "utt_id": utt, "path": str(ROOT_2021DF / "flac" / f"{utt}.flac"),
            "label": 0 if key == "bonafide" else 1,
            "attack": "bonafide" if key == "bonafide" else attack,
            "dataset": "asvspoof2021_df", "lang": "en", "split": "eval", "codec": "unknown",
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(MANIFEST_DIR / "asvspoof2021_df_eval.csv", index=False)
    log(f"  asvspoof2021_df_eval.csv: {len(df)} rows")

# -- In-the-Wild --
ROOT_ITW = DATA_RAW / "in_the_wild"
# Check if release_in_the_wild/ subdirectory exists (Kaggle structure)
ITW_AUDIO_ROOT = ROOT_ITW
if ROOT_ITW.exists():
    if (ROOT_ITW / "release_in_the_wild").exists():
        ITW_AUDIO_ROOT = ROOT_ITW / "release_in_the_wild"
        log(f"  In-the-Wild: audio in release_in_the_wild/ subdirectory")
if ROOT_ITW.exists() and (ROOT_ITW / "meta.csv").exists():
    log("Building In-the-Wild manifest...")
    rows = []
    for _, r in pd.read_csv(ROOT_ITW / "meta.csv").iterrows():
        fname = r["file"]
        label = r["label"]
        # Build path: try release_in_the_wild/ prefix if files not at root
        audio_path = ROOT_ITW / fname
        if not audio_path.exists():
            audio_path = ITW_AUDIO_ROOT / fname
        rows.append({
            "utt_id": fname.replace(".wav", ""),
            "path": str(audio_path),
            "label": 0 if label == "bonafide" else 1,
            "attack": "bonafide" if label == "bonafide" else "spoof",
            "dataset": "in_the_wild", "lang": "en", "split": "eval", "codec": "none",
        })
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(MANIFEST_DIR / "in_the_wild.csv", index=False)
    log(f"  in_the_wild.csv: {len(df)} rows")

# -- WaveFake --
ROOT_WF = DATA_RAW / "wavefake"
if ROOT_WF.exists():
    log("Building WaveFake manifest...")
    rows = []
    # Match multiple naming patterns: *_gen.wav, *_generated.wav, gen_*.wav
    seen = set()
    for pattern in ["*_gen.wav", "*_generated.wav", "gen_*.wav"]:
        for wav in sorted(ROOT_WF.rglob(pattern)):
            if wav not in seen:
                seen.add(wav)
                rows.append({
                    "utt_id": wav.stem, "path": str(wav), "label": 1,
                    "attack": wav.parent.name, "dataset": "wavefake",
                    "lang": "en", "split": "eval", "codec": "none",
                })
    # Try adding LJSpeech bonafide
    ljspeech = DATA_RAW / "LJSpeech"
    if ljspeech.exists():
        for row in rows[:]:
            ref = ljspeech / (row["utt_id"].replace("_gen", "") + ".wav")
            if ref.exists():
                rows.append({
                    "utt_id": ref.stem, "path": str(ref), "label": 0,
                    "attack": "bonafide", "dataset": "wavefake",
                    "lang": "en", "split": "eval", "codec": "none",
                })
    df = pd.DataFrame(rows, columns=COLUMNS).drop_duplicates(subset=["utt_id"])
    df.to_csv(MANIFEST_DIR / "wavefake.csv", index=False)
    log(f"  wavefake.csv: {len(df)} rows")

log("All manifests built")
mark_done("build_manifests")

In [ ]:
# ── 7. Build augmentation manifests (RIRs + MUSAN) ──
# Always rebuild -- avoids stale Windows paths from repo

# RIRs: find the data directory
rir_root = DATA_RAW / "RIRS_NOISES"
if not rir_root.exists() or not any(rir_root.iterdir()):
    # Try Kaggle inputs
    for kaggle_rir in [
        Path("/kaggle/input/rirs-noises"),
        Path("/kaggle/input/datasets/nhattruongdev/rirs-noises"),
        Path("/kaggle/input/musan-rirs"),
        Path("/kaggle/input/datasets/riosgonzalo/musan-rirs"),
    ]:
        if kaggle_rir.exists() and any(kaggle_rir.iterdir()):
            # Check for nested RIRS_NOISES/ directory
            nested = kaggle_rir / "RIRS_NOISES"
            if nested.exists() and any(nested.iterdir()):
                rir_root = DATA_RAW / "RIRS_NOISES"
                rir_root.parent.mkdir(parents=True, exist_ok=True)
                os.symlink(str(nested), str(rir_root), target_is_directory=True)
                log(f"  Symlinked RIRs: {rir_root} -> {nested}")
            elif (kaggle_rir / "simulated_rirs").exists() or (kaggle_rir / "pointsource_noises").exists():
                rir_root = DATA_RAW / "RIRS_NOISES"
                rir_root.parent.mkdir(parents=True, exist_ok=True)
                os.symlink(str(kaggle_rir), str(rir_root), target_is_directory=True)
                log(f"  Symlinked RIRs: {rir_root} -> {kaggle_rir}")
            break

rows = []
if rir_root.exists():
    # Try multiple possible RIR directory structures
    for rir_dir in ["real_rirs_isotropic_noises/real_rirs_isotropic_noises",
                    "real_rirs_isotropic_noises",
                    "simulated_rirs/mediumroom", "simulated_rirs/largeroom",
                    "simulated_rirs/smallroom", "simulated_rirs"]:
        d = rir_root / rir_dir
        if d.exists():
            files = list(d.rglob("*.wav"))
            rows.extend({"path": str(f)} for f in files)
            log(f"  {rir_dir}: {len(files)} files")
pd.DataFrame(rows).to_csv(MANIFEST_DIR / "rirs.csv", index=False)
log(f"RIRs manifest: {len(rows)} files")
if len(rows) == 0:
    log("RIRs not found -- reverb augmentation disabled")

# MUSAN
musan_root = DATA_RAW / "musan"
if not musan_root.exists() or not any(musan_root.iterdir()):
    for kaggle_musan in [
        Path("/kaggle/input/musan-rirs"),
        Path("/kaggle/input/datasets/riosgonzalo/musan-rirs"),
    ]:
        if not kaggle_musan.exists():
            continue
        # Check for nested musan/ directory
        musan_candidate = kaggle_musan / "musan"
        if musan_candidate.exists() and any(musan_candidate.iterdir()):
            musan_root.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(str(musan_candidate), str(musan_root), target_is_directory=True)
            log(f"  Symlinked MUSAN: {musan_root} -> {musan_candidate}")
            break
        # If musan/ subdir doesn't exist, try the root directly
        if (kaggle_musan / "music").exists() or (kaggle_musan / "noise").exists():
            musan_root.parent.mkdir(parents=True, exist_ok=True)
            os.symlink(str(kaggle_musan), str(musan_root), target_is_directory=True)
            log(f"  Symlinked MUSAN: {musan_root} -> {kaggle_musan}")
            break

rows = []
if musan_root.exists():
    for wav in musan_root.rglob("*.wav"):
        rows.append({"path": str(wav)})
pd.DataFrame(rows).to_csv(MANIFEST_DIR / "musan.csv", index=False)
log(f"MUSAN manifest: {len(rows)} files")
if len(rows) == 0:
    log("MUSAN not found -- noise augmentation disabled")

log("Augmentation manifests ready")
mark_done("aug_manifests")

## 8. Training (with checkpoint recovery + live HF upload)
Each model trains with automatic resume from last checkpoint.
Checkpoints are uploaded to HuggingFace after EVERY epoch (in background thread).
If session/network crashes, checkpoints are already on HF -- resume from new session.

**Expected times:** B1=1hr, B2=2hr, B3=3hr, B5=6hr, AuralGuard=8hr

In [ ]:
# ── B1 (LFCC-LCNN) ──
os.chdir("/kaggle/working/auralguard")
log("="*60)
log("B1 (LFCC-LCNN) -- estimated 1 hr")
log("="*60)

# Check for existing checkpoint
resume_epoch = get_resume_epoch("b1_lcnn")
if resume_epoch > 0:
    log(f"Resuming from epoch {resume_epoch}")
else:
    log("Starting fresh")

run_cmd(
    f"python scripts/train.py +experiment=b1_lcnn "
    f"data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv "
    f"data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv "
    f"data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv "
    f"data.augment.reverb.corpus=/kaggle/working/data/manifests/rirs.csv "
    f"data.augment.add_noise.corpus=/kaggle/working/data/manifests/musan.csv "
    f"data.num_workers=0 "
    f"+train.hf_upload.enabled=true "
    f"+train.hf_upload.repo={HF_REPO} "
    f"+train.hf_upload.upload_every_n_epochs=1",
    timeout_min=90, label="B1"
)

# Post-training: find best checkpoint and upload
b1_ckpt = Path("experiments/b1_lcnn/checkpoints/best.ckpt")
if b1_ckpt.exists():
    hf_upload(b1_ckpt, "checkpoints/b1_lcnn/best.ckpt")
    log("B1 best.ckpt uploaded to HF")
else:
    log("B1 best.ckpt not found -- training may have failed")
log("B1 DONE")
if b1_ckpt.exists():
    mark_done("train_b1")

In [ ]:
# ── B2 (RawNet2) ──
os.chdir("/kaggle/working/auralguard")
log("="*60)
log("B2 (RawNet2) -- estimated 2 hr")
log("="*60)

resume_epoch = get_resume_epoch("b2_rawnet2")
if resume_epoch > 0:
    log(f"Resuming from epoch {resume_epoch}")

run_cmd(
    f"python scripts/train.py +experiment=b2_rawnet2 "
    f"data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv "
    f"data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv "
    f"data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv "
    f"data.augment.reverb.corpus=/kaggle/working/data/manifests/rirs.csv "
    f"data.augment.add_noise.corpus=/kaggle/working/data/manifests/musan.csv "
    f"data.num_workers=0 "
    f"+train.hf_upload.enabled=true "
    f"+train.hf_upload.repo={HF_REPO} "
    f"+train.hf_upload.upload_every_n_epochs=1",
    timeout_min=150, label="B2"
)

b2_ckpt = Path("experiments/b2_rawnet2/checkpoints/best.ckpt")
if b2_ckpt.exists():
    hf_upload(b2_ckpt, "checkpoints/b2_rawnet2/best.ckpt")
log("B2 DONE")
if b2_ckpt.exists():
    mark_done("train_b2")

In [ ]:
# ── B3 (AASIST) ──
os.chdir("/kaggle/working/auralguard")
log("="*60)
log("B3 (AASIST) -- estimated 3 hr")
log("="*60)

resume_epoch = get_resume_epoch("b3_aasist")
if resume_epoch > 0:
    log(f"Resuming from epoch {resume_epoch}")

run_cmd(
    f"python scripts/train.py +experiment=b3_aasist "
    f"data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv "
    f"data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv "
    f"data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv "
    f"data.augment.reverb.corpus=/kaggle/working/data/manifests/rirs.csv "
    f"data.augment.add_noise.corpus=/kaggle/working/data/manifests/musan.csv "
    f"data.num_workers=0 "
    f"+train.hf_upload.enabled=true "
    f"+train.hf_upload.repo={HF_REPO} "
    f"+train.hf_upload.upload_every_n_epochs=1",
    timeout_min=210, label="B3"
)

b3_ckpt = Path("experiments/b3_aasist/checkpoints/best.ckpt")
if b3_ckpt.exists():
    hf_upload(b3_ckpt, "checkpoints/b3_aasist/best.ckpt")
log("B3 DONE")
if b3_ckpt.exists():
    mark_done("train_b3")

In [ ]:
# ── B5 (WavLM + AASIST + OCSoftmax) ──
os.chdir("/kaggle/working/auralguard")
log("="*60)
log("B5 (WavLM+OCS) -- estimated 6 hr")
log("="*60)

resume_epoch = get_resume_epoch("b5_wavlm_ocs")
if resume_epoch > 0:
    log(f"Resuming from epoch {resume_epoch}")

run_cmd(
    f"python scripts/train.py +experiment=b5_wavlm_ocs "
    f"data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv "
    f"data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv "
    f"data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv "
    f"data.augment.reverb.corpus=/kaggle/working/data/manifests/rirs.csv "
    f"data.augment.add_noise.corpus=/kaggle/working/data/manifests/musan.csv "
    f"data.num_workers=0 "
    f"+train.hf_upload.enabled=true "
    f"+train.hf_upload.repo={HF_REPO} "
    f"+train.hf_upload.upload_every_n_epochs=1",
    timeout_min=390, label="B5"
)

b5_ckpt = Path("experiments/b5_wavlm_ocs/checkpoints/best.ckpt")
if b5_ckpt.exists():
    hf_upload(b5_ckpt, "checkpoints/b5_wavlm_ocs/best.ckpt")
log("B5 DONE")
if b5_ckpt.exists():
    mark_done("train_b5")

In [ ]:
# ── AuralGuard (full model) ──
os.chdir("/kaggle/working/auralguard")
log("="*60)
log("AuralGuard (full) -- estimated 8 hr")
log("="*60)

resume_epoch = get_resume_epoch("auralguard")
if resume_epoch > 0:
    log(f"Resuming from epoch {resume_epoch}")

run_cmd(
    f"python scripts/train.py +experiment=auralguard "
    f"data.manifests.train=/kaggle/working/data/manifests/asvspoof2019_la_train.csv "
    f"data.manifests.dev=/kaggle/working/data/manifests/asvspoof2019_la_dev.csv "
    f"data.manifests.eval=/kaggle/working/data/manifests/asvspoof2019_la_eval.csv "
    f"data.augment.reverb.corpus=/kaggle/working/data/manifests/rirs.csv "
    f"data.augment.add_noise.corpus=/kaggle/working/data/manifests/musan.csv "
    f"data.num_workers=0 "
    f"+train.hf_upload.enabled=true "
    f"+train.hf_upload.repo={HF_REPO} "
    f"+train.hf_upload.upload_every_n_epochs=1",
    timeout_min=510, label="AuralGuard"
)

ag_ckpt = Path("experiments/auralguard/checkpoints/best.ckpt")
if ag_ckpt.exists():
    hf_upload(ag_ckpt, "checkpoints/auralguard/best.ckpt")
log("AuralGuard DONE")
if ag_ckpt.exists():
    mark_done("train_auralguard")

## 9. In-Domain Evaluation

In [ ]:
# ── Evaluate all models (in-domain) ──
os.chdir("/kaggle/working/auralguard")
log("Evaluating all models on in-domain test set...")

model_dirs = sorted([p for p in Path("experiments").iterdir() if p.is_dir()])
log(f"Found {len(model_dirs)} experiment(s): {[p.name for p in model_dirs]}")

for exp_dir in tqdm(model_dirs, desc="In-domain eval"):
    ckpt = exp_dir / "checkpoints" / "best.ckpt"
    if not ckpt.exists():
        log(f"  [skip] {exp_dir.name} -- no best.ckpt")
        continue
    out_dir = exp_dir / "eval_results"
    log(f"  Evaluating {exp_dir.name}...")
    run_cmd([
        "python", "scripts/evaluate.py",
        f"--ckpt={ckpt}",
        f"--out={out_dir}",
    ], timeout_min=30)

log("All in-domain evaluations complete")
mark_done("eval_in_domain")

## 10. Zero-Shot Evaluation (all unseen datasets)

In [ ]:
# ── Zero-shot eval across all unseen datasets ──
os.chdir("/kaggle/working/auralguard")
log("Evaluating all models on zero-shot corpora...")

model_dirs = sorted([p for p in Path("experiments").iterdir() if p.is_dir()])

for exp_dir in tqdm(model_dirs, desc="Zero-shot eval"):
    ckpt = exp_dir / "checkpoints" / "best.ckpt"
    if not ckpt.exists():
        log(f"  [skip] {exp_dir.name} -- no best.ckpt")
        continue
    out_dir = exp_dir / "zeroshot_results"
    log(f"  Zero-shot eval for {exp_dir.name}...")
    run_cmd([
        "python", "scripts/eval_all_zeroshot.py",
        f"--ckpt={ckpt}",
        f"--out={out_dir}",
    ], timeout_min=60)

log("All zero-shot evaluations complete")
mark_done("eval_zero_shot")

## 11. Generate Paper Figures

In [ ]:
# ── Generate all paper figures ──
os.chdir("/kaggle/working/auralguard")

import json, math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.dpi": 150, "font.size": 11})
FIGS = Path("/kaggle/working/paper_figures")
FIGS.mkdir(parents=True, exist_ok=True)

model_order = ["b1_lcnn", "b2_rawnet2", "b3_aasist", "b5_wavlm_ocs", "auralguard"]
display_names = {
    "b1_lcnn": "B1 (LFCC-LCNN)",
    "b2_rawnet2": "B2 (RawNet2)",
    "b3_aasist": "B3 (AASIST)",
    "b5_wavlm_ocs": "B5 (WavLM+OCS)",
    "auralguard": "AuralGuard",
}
colors = plt.cm.tab10(np.linspace(0, 1, len(model_order)))

# [1/5] Training curves
log("[1/5] Training curves...")
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for idx, name in enumerate(tqdm(model_order, desc="Curves")):
        log_dir = Path("experiments") / name / "logs"
        if not log_dir.exists(): continue
        try:
            ea = EventAccumulator(str(log_dir))
            ea.Reload()
            tags = ea.Tags().get("scalars", [])
            for ax, tag, ylabel in zip(axes, ["train/loss", "eval/eer"], ["Loss", "EER (%)"]):
                if tag in tags:
                    events = ea.Scalars(tag)
                    ax.plot([e.step for e in events], [e.value for e in events],
                            label=display_names.get(name, name), color=colors[idx], linewidth=1.5)
                    ax.set_xlabel("Step"); ax.set_ylabel(ylabel)
                    ax.legend(fontsize=8, loc="upper right"); ax.grid(True, alpha=0.3)
        except Exception as e:
            log(f"  [skip] {name}: {e}")
    plt.tight_layout()
    fig.savefig(FIGS / "training_curves.png", bbox_inches="tight"); plt.close()
    log(f"  -> {FIGS / 'training_curves.png'}")
except ImportError:
    log("  [skip] tensorboard not installed")

# [2/5] EER bar chart (all datasets)
log("[2/5] EER bar chart...")
try:
    rows = []
    for name in model_order:
        for suffix in ["eval_results", "zeroshot_results"]:
            res_file = Path("experiments") / name / suffix / "results.json"
            if res_file.exists():
                data = json.loads(res_file.read_text())
                for ds_name, metrics in data.items():
                    rows.append({"model": name, "dataset": ds_name, "eer": metrics["eer"]})
    if rows:
        df = pd.DataFrame(rows)
        dataset_order = ["in_domain_eval", "in_the_wild", "wavefake", "mlaad",
                         "asvspoof2021_la", "asvspoof2021_df"]
        df = df[df["dataset"].isin(dataset_order)]
        df["dataset"] = pd.Categorical(df["dataset"], categories=dataset_order, ordered=True)
        df["model"] = pd.Categorical(df["model"], categories=model_order, ordered=True)
        df = df.sort_values(["dataset", "model"])
        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(dataset_order)); bw = 0.15
        for i, name in enumerate(model_order):
            subset = df[df["model"] == name]
            vals = [subset[subset["dataset"] == d]["eer"].values[0]
                    if len(subset[subset["dataset"] == d]) > 0 else 0 for d in dataset_order]
            ax.bar(x + (i - len(model_order)/2 + 0.5) * bw, [v*100 for v in vals], bw,
                   label=display_names.get(name, name), color=colors[i])
        ax.set_xticks(x)
        ax.set_xticklabels([d.replace("in_domain_eval", "In-Domain")
                             .replace("in_the_wild", "In-the-Wild")
                             .replace("wavefake", "WaveFake")
                             .replace("mlaad", "MLAAD")
                             .replace("asvspoof2021_la", "ASVspoof21 LA")
                             .replace("asvspoof2021_df", "ASVspoof21 DF")])
        ax.set_ylabel("EER (%)"); ax.set_title("In-Domain & Zero-Shot EER Comparison")
        ax.legend(fontsize=8, loc="upper left"); ax.grid(True, alpha=0.3, axis="y")
        plt.tight_layout()
        fig.savefig(FIGS / "eer_comparison.png", bbox_inches="tight"); plt.close()
        log(f"  -> {FIGS / 'eer_comparison.png'}")
except Exception as e:
    log(f"  [skip] EER bar chart: {e}")

# [3/5] DET curves
log("[3/5] DET curves...")
try:
    from sklearn.metrics import roc_curve
    fig, ax = plt.subplots(figsize=(6, 6))
    for name, c in zip(model_order, colors):
        res_file = Path("experiments") / name / "eval_results" / "results.json"
        if not res_file.exists(): continue
        data = json.loads(res_file.read_text())
        in_domain = data.get("in_domain_eval", {})
        if "scores" not in in_domain or "labels" not in in_domain: continue
        scores, labels = np.array(in_domain["scores"]), np.array(in_domain["labels"])
        fpr, fnr, _ = roc_curve(labels, scores, pos_label=1)
        mask = (fpr > 1e-5) & (fnr > 1e-5)
        ax.loglog(fpr[mask], fnr[mask], label=display_names.get(name, name), color=c, linewidth=1.5)
    ax.set_xlabel("False Alarm Rate (FAR)"); ax.set_ylabel("False Rejection Rate (FRR)")
    ax.set_title("DET Curve -- In-Domain Evaluation")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    fig.savefig(FIGS / "det_curve.png", bbox_inches="tight"); plt.close()
    log(f"  -> {FIGS / 'det_curve.png'}")
except Exception as e:
    log(f"  [skip] DET curve: {e}")

# [4/5] Score distribution
log("[4/5] Score distribution...")
try:
    res_file = Path("experiments") / "auralguard" / "eval_results" / "results.json"
    if res_file.exists():
        data = json.loads(res_file.read_text())
        in_domain = data.get("in_domain_eval", {})
        if "scores" in in_domain and "labels" in in_domain:
            scores, labels = np.array(in_domain["scores"]), np.array(in_domain["labels"])
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.hist(scores[labels==0], bins=80, alpha=0.6, label="Bona-fide", color="green", density=True)
            ax.hist(scores[labels==1], bins=80, alpha=0.6, label="Spoof", color="red", density=True)
            ax.axvline(0, color="black", linestyle="--", alpha=0.5, label="Decision boundary")
            ax.set_xlabel("Spoof Score"); ax.set_ylabel("Density")
            ax.set_title("AuralGuard -- Score Distribution (In-Domain Eval)")
            ax.legend(); ax.grid(True, alpha=0.3)
            plt.tight_layout()
            fig.savefig(FIGS / "score_distribution.png", bbox_inches="tight"); plt.close()
            log(f"  -> {FIGS / 'score_distribution.png'}")
except Exception as e:
    log(f"  [skip] Score distribution: {e}")

# [5/5] Robustness plots (EER vs dataset)
log("[5/5] Robustness heatmap...")
try:
    all_rows = []
    for name in model_order:
        for suffix in ["eval_results", "zeroshot_results"]:
            res_file = Path("experiments") / name / suffix / "results.json"
            if res_file.exists():
                data = json.loads(res_file.read_text())
                for ds, m in data.items():
                    all_rows.append({"model": display_names.get(name, name),
                                     "dataset": ds, "eer": m["eer"] * 100})
    if all_rows:
        df = pd.DataFrame(all_rows)
        pivot = df.pivot(index="model", columns="dataset", values="eer")
        fig, ax = plt.subplots(figsize=(10, 4))
        im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=30)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=9)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=9)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8,
                            color="white" if val > 15 else "black")
        plt.colorbar(im, ax=ax, label="EER (%)")
        ax.set_title("Robustness Heatmap -- EER (%) across datasets")
        plt.tight_layout()
        fig.savefig(FIGS / "robustness_heatmap.png", bbox_inches="tight"); plt.close()
        log(f"  -> {FIGS / 'robustness_heatmap.png'}")
except Exception as e:
    log(f"  [skip] Robustness heatmap: {e}")

log("All figures generated")
mark_done("figures")

## 12. Export Results Table

In [ ]:
# ── Export results as markdown + CSV ──
os.chdir("/kaggle/working/auralguard")
log("Exporting results table...")

rows = []
for name in model_order:
    for suffix, label in [("eval_results", "In-Domain"), ("zeroshot_results", "Zero-Shot")]:
        res_file = Path("experiments") / name / suffix / "results.json"
        if res_file.exists():
            data = json.loads(res_file.read_text())
            for ds_name, m in data.items():
                rows.append({
                    "Model": display_names.get(name, name),
                    "Dataset": ds_name,
                    "EER": f"{m['eer']:.4f}",
                    "EER CI95": f"[{m.get('eer_ci95', [0,0])[0]:.4f}, {m.get('eer_ci95', [0,0])[1]:.4f}]",
                    "AUROC": f"{m['auroc']:.4f}",
                    "min t-DCF": f"{m['min_tdcf']:.4f}",
                    "F1": f"{m.get('f1', 0):.4f}",
                    "Bal. Acc": f"{m.get('balanced_accuracy', 0):.4f}",
                })

if rows:
    df = pd.DataFrame(rows)
    md = df.to_markdown(index=False)
    (FIGS / "results_table.md").write_text(md)
    df.to_csv(FIGS / "results_table.csv", index=False)
    log(f"Results table: {len(rows)} rows")
    print("\n" + md + "\n")
else:
    log("No results found")
mark_done("results_table")

## 13. Upload Everything to HuggingFace
Checkpoints, results, figures, and metrics are pushed to `MoshinAli/auralguard-checkpoints`.
This enables cross-account recovery: just clone/download from HF in a new session.

In [ ]:
# ── Upload all results + figures + checkpoints to HuggingFace ──
os.chdir("/kaggle/working/auralguard")
log("="*60)
log("Uploading everything to HuggingFace Hub...")
log("="*60)

# 1. Upload checkpoints for all models
log("\n--- Checkpoints ---")
for name in model_order:
    ckpt_dir = Path("experiments") / name / "checkpoints"
    for ckpt_name in ["best.ckpt", "last.ckpt"]:
        ckpt_path = ckpt_dir / ckpt_name
        if ckpt_path.exists():
            hf_upload(ckpt_path, f"checkpoints/{name}/{ckpt_name}")

# 2. Upload evaluation results
log("\n--- Evaluation Results ---")
for name in model_order:
    for suffix in ["eval_results", "zeroshot_results"]:
        res_file = Path("experiments") / name / suffix / "results.json"
        if res_file.exists():
            hf_upload(res_file, f"results/{name}/{suffix}/results.json")

# 3. Upload paper figures
log("\n--- Paper Figures ---")
for fig in Path("/kaggle/working/paper_figures").glob("*"):
    if fig.is_file():
        hf_upload(fig, f"figures/{fig.name}")

# 4. Upload training logs (TensorBoard)
log("\n--- Training Logs ---")
for name in model_order:
    log_dir = Path("experiments") / name / "logs"
    if log_dir.exists():
        for event_file in log_dir.rglob("events.out.tfevents.*"):
            hf_upload(event_file, f"logs/{name}/{event_file.name}")

log("\n" + "="*60)
log("All uploads complete!")
log("="*60)
log(f"HF Repo: https://huggingface.co/{HF_REPO}")
log("You can now pull checkpoints from any new Kaggle session or account.")
mark_done("hf_upload_all")

## 14. Upload Model to HuggingFace for Deployment
Creates a dedicated HF model repo with:
- `best.ckpt` — trained model weights + config (ready for inference)
- `README.md` — model card with usage instructions
- The HF Space auto-loads from this repo at startup

In [ ]:
# ── Upload AuralGuard model to dedicated HF repo for deployment ──
os.chdir("/kaggle/working/auralguard")
log("="*60)
log("Uploading model to HuggingFace for deployment...")
log("="*60)

from huggingface_hub import HfApi, login
import json as _json

# Model repo name (separate from checkpoints repo)
MODEL_REPO = "MoshinAli/auralguard"

# 1. Create model repo
api = HfApi()
try:
    api.repo_info(MODEL_REPO)
    log(f"Repo exists: {MODEL_REPO}")
except Exception:
    api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
    log(f"Created repo: {MODEL_REPO}")

# 2. Find best AuralGuard checkpoint
best_ckpt = Path("experiments/auralguard/checkpoints/best.ckpt")
if not best_ckpt.exists():
    log("WARNING: best.ckpt not found for AuralGuard")
    log("Trying last.ckpt...")
    best_ckpt = Path("experiments/auralguard/checkpoints/last.ckpt")

if best_ckpt.exists():
    # 3. Create slim checkpoint (weights + config only, no optimizer state)
    import torch
    ckpt = torch.load(str(best_ckpt), map_location="cpu", weights_only=False)
    slim_ckpt = {
        "model": ckpt["model"],
        "cfg": ckpt["cfg"],
        "epoch": ckpt.get("epoch", 0),
        "dev_eer": ckpt.get("dev_eer", 0),
    }
    slim_path = Path("/kaggle/working/auralguard_deploy/best.ckpt")
    slim_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(slim_ckpt, str(slim_path))
    log(f"Slim checkpoint saved: {slim_path} ({slim_path.stat().st_size / 1e6:.1f} MB)")

    # 4. Generate model card (README.md)
    cfg = ckpt["cfg"]
    model_cfg = cfg.get("model", {})
    data_cfg = cfg.get("data", {})
    train_cfg = cfg.get("train", {})
    dev_eer = ckpt.get("dev_eer", 0)
    epoch = ckpt.get("epoch", 0)

    # Read eval results if available
    eval_results = {}
    for suffix in ["eval_results", "zeroshot_results"]:
        res_file = Path("experiments/auralguard") / suffix / "results.json"
        if res_file.exists():
            eval_results[suffix] = _json.loads(res_file.read_text())

    results_section = ""
    if eval_results:
        results_section = "\n## Evaluation Results\n\n"
        for suffix, data in eval_results.items():
            results_section += f"### {suffix.replace('_', ' ').title()}\n\n"
            results_section += "| Dataset | EER | AUROC | min t-DCF |\n"
            results_section += "|---------|-----|-------|-----------|\n"
            for ds_name, m in data.items():
                results_section += f"| {ds_name} | {m.get('eer', 0):.4f} | {m.get('auroc', 0):.4f} | {m.get('min_tdcf', 0):.4f} |\n"
            results_section += "\n"

    model_card = f"""---
tags:
  - audio-deepfake-detection
  - anti-spoofing
  - synthetic-speech-detection
  - asvspoof
  - wavlm
  - aasist
  - one-class-learning
license: mit
language:
  - en
library_name: auralguard
pipeline_tag: audio-classification
model-index:
  - name: AuralGuard
    results:
      - task:
          type: audio-classification
          name: AI-Generated Speech Detection
        dataset:
          type: asvspoof2019_la
          name: ASVspoof 2019 LA
        metrics:
          - type: eer
            value: {dev_eer:.4f}
            name: Equal Error Rate
---

# AuralGuard

Multi-view one-class detection of AI-generated speech.

## Model Description

AuralGuard is a deepfake speech detector that combines:
- **SSL Frontend**: WavLM-Large with learnable layer attention
- **Artifact Frontend**: LFCC + modified group-delay + CQT phase features
- **Fusion**: Gated cross-attention between views
- **Backend**: AASIST graph-attention readout
- **Training**: OC-Softmax + supervised contrastive loss

## Usage

### Quick Start (Python)

```python
from huggingface_hub import hf_hub_download
from auralguard.inference.predict import Detector

# Download checkpoint
ckpt_path = hf_hub_download(
    repo_id="MoshinAli/auralguard",
    filename="best.ckpt"
)

# Load detector
detector = Detector(ckpt_path)

# Predict
result = detector.predict_file("audio.wav")
print(result)
# {'verdict': 'ai_generated', 'p_ai_generated': 0.92, 'confidence': 'high', ...}
```

### API Usage

```bash
curl -X POST https://MoshinAli-auralguard.hf.space/api/detect \
  -F "file=@audio.wav"
```

### CLI Usage

```bash
pip install -e .
auralguard-predict --audio audio.wav --ckpt best.ckpt
```

## Input

- **Format**: WAV, FLAC, MP3, OGG, M4A, OPUS, WEBM
- **Sample rate**: Any (auto-resampled to 16 kHz)
- **Duration**: Any (sliding window 4s, hop 2s, mean pooling)

## Output

```json
{
  "verdict": "ai_generated",
  "p_ai_generated": 0.9234,
  "confidence": "high",
  "score": 2.5432,
  "threshold": 0.5,
  "windows": 3,
  "model_version": "auralguard",
  "sample_rate": 16000,
  "duration_s": 6.5
}
```

## Training Details

- **Dataset**: ASVspoof 2019 LA
- **Epochs**: {epoch}
- **Dev EER**: {dev_eer:.4f}
- **Optimizer**: AdamW (lr={train_cfg.get('optimizer', {}).get('lr', 'N/A')})
- **Hardware**: Single GPU (P100/T4)

{results_section}
## Citation

```bibtex
@article{{auralguard2024,
  title={{AuralGuard: Multi-View One-Class Learning for Generalizable Detection of AI-Generated Speech in the Wild}},
  author={{AuralGuard Authors}},
  year={{2024}}
}}
```

## License

MIT
"""

    readme_path = slim_path.parent / "README.md"
    readme_path.write_text(model_card)
    log(f"Model card saved: {readme_path}")

    # 5. Upload to HuggingFace
    log("Uploading best.ckpt...")
    api.upload_file(
        path_or_fileobj=str(slim_path),
        path_in_repo="best.ckpt",
        repo_id=MODEL_REPO,
        repo_type="model",
    )
    log(f"Uploaded best.ckpt to {MODEL_REPO}")

    log("Uploading README.md...")
    api.upload_file(
        path_or_fileobj=str(readme_path),
        path_in_repo="README.md",
        repo_id=MODEL_REPO,
        repo_type="model",
    )
    log(f"Uploaded README.md to {MODEL_REPO}")

    # 6. Also export ONNX if possible
    log("Trying ONNX export...")
    try:
        from auralguard.models import build_model
        model = build_model(cfg["model"])
        model.load_state_dict(ckpt["model"])
        model.eval()
        dummy = torch.randn(1, 64000)
        onnx_path = slim_path.parent / "model.onnx"
        torch.onnx.export(
            model, dummy, str(onnx_path),
            input_names=["waveform"], output_names=["score"],
            dynamic_axes={"waveform": {1: "samples"}},
            opset_version=17,
        )
        api.upload_file(
            path_or_fileobj=str(onnx_path),
            path_in_repo="model.onnx",
            repo_id=MODEL_REPO,
            repo_type="model",
        )
        log(f"Uploaded model.onnx to {MODEL_REPO}")
    except Exception as e:
        log(f"ONNX export skipped: {e}")

    log("\n" + "="*60)
    log(f"MODEL DEPLOYED TO: https://huggingface.co/{MODEL_REPO}")
    log("="*60)
    log("")
    log("To deploy as HF Space:")
    log(f"  1. Create a new HF Space (Docker SDK)")
    log(f"  2. Set env var MODEL_REPO={MODEL_REPO}")
    log(f"  3. Set env var MODEL_FILE=best.ckpt")
    log(f"  4. The Space auto-downloads and serves the model")
else:
    log("No checkpoint found -- skipping model upload")
mark_done("model_deploy")

## 15. Package for Local Download

In [ ]:
# ── Package results + figures for Kaggle download ──
import tarfile

tar_path = "/kaggle/working/auralguard_results.tar.gz"
log(f"Packaging -> {tar_path}")

paths_to_add = []
for ckpt in Path("experiments").rglob("best.ckpt"):
    paths_to_add.append((str(ckpt), str(ckpt.relative_to("/kaggle/working"))))
for res in Path("experiments").rglob("results.json"):
    paths_to_add.append((str(res), str(res.relative_to("/kaggle/working"))))
for fig in Path("/kaggle/working/paper_figures").glob("*"):
    if fig.is_file():
        paths_to_add.append((str(fig), f"paper_figures/{fig.name}"))

with tarfile.open(tar_path, "w:gz") as tar:
    for src, arcname in tqdm(paths_to_add, desc="Tar"):
        tar.add(src, arcname=arcname)

size = Path(tar_path).stat().st_size
log(f"Package: {size / 1e6:.1f} MB")
log("Download via Kaggle sidebar -> Output -> auralguard_results.tar.gz")
mark_done("package")

## 16. Recovery — Pull from HuggingFace & Resume

**When to use:** Session crashed, network died, 30h limit hit, or switching accounts.

### Steps:
1. Open a **new Kaggle notebook** (enable Internet + GPU)
2. Run Cell 1 (helpers) and Cell 2 (HF setup) and Cell 5 (HF login)
3. Run the cell below to pull checkpoints
4. Run Cell 4d (install AuralGuard)
5. Re-run the training cell for the model you want to continue
   - It auto-detects the pulled `last.ckpt` and resumes from that epoch

In [ ]:
# ── RECOVERY: Pull checkpoints from HuggingFace ──
# Run this AFTER cells 1, 2, 5 (helpers + HF login)

HF_REPO = "MoshinAli/auralguard-checkpoints"

from huggingface_hub import hf_hub_download, HfApi
api = HfApi()

# List what's on HF
files = api.list_repo_files(HF_REPO)
print(f"Files on HuggingFace ({len(files)} total):")
for f in sorted(files):
    print(f"  {f}")

# Download all checkpoints to correct local paths
downloaded = 0
for f in sorted(files):
    if not f.endswith(".ckpt"):
        continue
    local_dir = Path("/kaggle/working/auralguard") / Path(f).parent
    local_dir.mkdir(parents=True, exist_ok=True)
    hf_hub_download(repo_id=HF_REPO, filename=f, local_dir=str(Path("/kaggle/working/auralguard")))
    print(f"  Downloaded: {f}")
    downloaded += 1

print(f"\nDownloaded {downloaded} checkpoints")

# Show what epoch each model is at
import torch
print("\nResume status:")
for exp in ["b1_lcnn", "b2_rawnet2", "b3_aasist", "b5_wavlm_ocs", "auralguard"]:
    last = Path(f"/kaggle/working/auralguard/experiments/{exp}/checkpoints/last.ckpt")
    best = Path(f"/kaggle/working/auralguard/experiments/{exp}/checkpoints/best.ckpt")
    if last.exists():
        ckpt = torch.load(str(last), map_location="cpu", weights_only=False)
        print(f"  {exp:20s} -> resume from epoch {ckpt.get('epoch', -1) + 1}  (best EER={ckpt.get('dev_eer', 0):.4f})")
    elif best.exists():
        ckpt = torch.load(str(best), map_location="cpu", weights_only=False)
        print(f"  {exp:20s} -> has best.ckpt at epoch {ckpt.get('epoch', -1)}  (EER={ckpt.get('dev_eer', 0):.4f})")
    else:
        print(f"  {exp:20s} -> no checkpoint found, will train from scratch")

print("\nNow re-run the training cell for the model you want to continue.")
print("The trainer auto-detects last.ckpt and resumes from the correct epoch.")

In [ ]:
# ── QUICK RESUME: Pull single model + re-run training ──
# Uncomment the model you want to resume, then run this cell.

RESUME_MODEL = "auralguard"  # Change to: b1_lcnn, b2_rawnet2, b3_aasist, b5_wavlm_ocs, auralguard

HF_REPO = "MoshinAli/auralguard-checkpoints"
from huggingface_hub import hf_hub_download
import torch

# Pull best.ckpt and last.ckpt for this model
exp_dir = Path(f"/kaggle/working/auralguard/experiments/{RESUME_MODEL}/checkpoints")
exp_dir.mkdir(parents=True, exist_ok=True)

for ckpt_name in ["best.ckpt", "last.ckpt"]:
    repo_path = f"checkpoints/{RESUME_MODEL}/{ckpt_name}"
    try:
        hf_hub_download(repo_id=HF_REPO, filename=repo_path, local_dir=str(Path("/kaggle/working/auralguard")))
        print(f"Downloaded: {repo_path}")
    except Exception as e:
        print(f"Skip {repo_path}: {e}")

# Check epoch
last_ckpt = exp_dir / "last.ckpt"
if last_ckpt.exists():
    ckpt = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    epoch = ckpt.get("epoch", -1)
    eer = ckpt.get("dev_eer", 0)
    print(f"\nReady to resume {RESUME_MODEL} from epoch {epoch + 1} (best EER={eer:.4f})")
    print(f"\nNow run the training cell for {RESUME_MODEL} -- it will auto-resume.")
else:
    print(f"\nNo checkpoint found for {RESUME_MODEL}")